# Rotational interaction

Mode that creates ghost atoms following interactions and guiding interacted atoms to match rotation of the controller:

<video controls src="./assets/rotational_interaction.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation
from nanover.trajectory import FrameData
from nanover.mdanalysis import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("trypsin_benzamidine.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: rotational interaction")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities, SceneObjectsUtility

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_interaction_modes()
utilities.use_transform_handles()

In [3]:
structure_atoms = universe.select_atoms("not resname BEN")
molecule_atoms = universe.select_atoms("resname BEN and not name H*")

utilities.selections.update_selection("root", renderer="cartoon")
utilities.selections.update_selection("ligand", renderer="liquorice", particle_ids=universe.select_atoms("resname BEN").atoms.indices)

## Ghosts

In [4]:
import numpy as np
import MDAnalysis as mda
from nanover.utilities.transforms import Transform


def make_ghost(key, atoms, positions=None):
    # extract ghost molecule
    ghost_universe = mda.Merge(atoms)
    ghost_positions = positions if positions is not None else ghost_universe.atoms.positions / 10  # angstrom -> nm
    ghost_bond_pairs = ghost_universe.bonds.indices

    # normalise around centroid, determine bounding radius
    centroid = np.mean(ghost_positions, axis=0)
    ghost_positions -= centroid

    # transform + handle for manipulating it
    utilities.transforms.update_transform(f"ghost.{key}", transform=Transform.from_translation(centroid), parent="simulation")

    visuals = SceneObjectsUtility.from_runner(imd_runner)

    # add transparent spheres and lines to scene at positions relative to nanotube in first frame:
    for i, position in enumerate(ghost_positions):
        visuals.update_shape(f"ghost.{key}.{i}", position=position, size=0.1, color=[1.0, 1.0, 1.0, 0.5], parent=f"ghost.{key}")
    for i, (a, b) in enumerate(ghost_bond_pairs):
        visuals.update_line(f"ghost.{key}.{i}", positions=ghost_positions[[a, b]], size=0.05, color=[1.0, 1.0, 1.0, 0.5], parent=f"ghost.{key}")

    return f"ghost.{key}", ghost_positions, visuals

In [5]:
from dataclasses import dataclass

import numpy.typing as npt

from nanover.utilities.transforms import state_transform_from_matrix, matrix_from_state_transform
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode

def get_cursor_id_from_interaction(interaction: ParticleInteraction):
    owner_id = interaction.properties.get("owner.id", "")
    hand = interaction.properties.get("label", "hand.").removeprefix("hand.")
    return f"cursor.{owner_id}.{hand}"


@dataclass(kw_only=True)
class Grab:
    ghost_id: str
    indices: list[int]
    positions: npt.NDArray
    offset_matrix: npt.NDArray
    visuals: SceneObjectsUtility
    follower: GhostFollowerAgent

cursor_grabs: dict[str, Grab] = {}


def cursor_in_object_parent_matrix(cursor: dict, object: str):
    """
    Return the matrix of the cursor relative to the parent of given transform.
    """

    object_parent = utilities.transforms.get_parent(object, default="root")
    root_to_parent = utilities.transforms.fetch_transform_root(
        object_parent
    ).parent_to_local_matrix
    cursor_to_root = Transform.from_state_cursor(cursor).local_to_parent_matrix
    return root_to_parent @ cursor_to_root

def filter_matrix_update(next_matrix, prev_matrix, handle: dict):
    """
    Return the next matrix with any translation/rotation/scale disallowed by the handle reverted to those of the
    previous matrix.
    """

    prev_pose = state_transform_from_matrix(prev_matrix)
    next_pose = list(state_transform_from_matrix(next_matrix))

    if not handle.get("translate", False):
        next_pose[0:3] = prev_pose[0:3]
    if not handle.get("rotate", False):
        next_pose[3:7] = prev_pose[3:7]
    if not handle.get("scale", False):
        next_pose[7:10] = prev_pose[7:10]

    return matrix_from_state_transform(next_pose)

class RotationalInteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        cursor_id = get_cursor_id_from_interaction(interaction)

        with imd_runner.app_server.state_dictionary.lock_content() as state:
            cursor = state.get(cursor_id)

        if cursor is None:
            return

        positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions[interaction.particles]
        ghost_id, ghost_positions, visuals = make_ghost(cursor_id, universe.atoms[interaction.particles], positions)

        # cursor matrix relative to object parent
        cursor_in_parent = cursor_in_object_parent_matrix(cursor, ghost_id)
        # object matrix relative to object parent
        object_in_parent = utilities.transforms.fetch_transform(ghost_id).local_to_parent_matrix
        # matrix transforming cursor to object
        offset_matrix = np.linalg.inv(cursor_in_parent) @ object_in_parent

        follower = GhostFollowerAgent.from_runner(imd_runner)
        grab = Grab(
            ghost_id=ghost_id,
            indices=[int(i) for i in interaction.particles],
            positions=ghost_positions,
            offset_matrix=offset_matrix,
            visuals=visuals,
            follower=follower,
        )

        follower.grab = grab
        follower.cursor_id = cursor_id
        follower.visuals = visuals
        follower.start()

        cursor_grabs[cursor_id] =  grab

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        cursor_id = get_cursor_id_from_interaction(interaction)
        # release grabbed
        grab = cursor_grabs.pop(cursor_id, None)
        if grab is not None:
            grab.visuals.clear()
            grab.follower.close()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        # if this cursor has grabbed an object, update the object pose from cursor pose
        grabbed = cursor_grabs.get(key, None)
        if grabbed is not None:
            object_parent = utilities.transforms.get_parent(
                grabbed.ghost_id, default="root"
            )
            # cursor matrix relative to object parent
            cursor_in_parent = cursor_in_object_parent_matrix(cursor, grabbed.ghost_id)
            # object matrix relative to object parent
            object_in_parent = cursor_in_parent @ grabbed.offset_matrix

            utilities.transforms.update_transform(
                grabbed.ghost_id,
                transform=Transform.from_local_to_parent_matrix(object_in_parent),
                parent=object_parent,
            )

utilities.add_interaction_mode(RotationalInteractMode, "rotational interaction", icon="🔃")

## Ghost follower

In [6]:
from nanover.jupyter import ImdAgent
from nanover.imd import ParticleInteraction


class GhostFollowerAgent(ImdAgent):
    cursor_id = ""
    grab = None
    visuals = None

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        grab = self.grab
        cursor_id = self.cursor_id

        # target positions are original ghost positions transformed by ghost transform
        target_positions = utilities.transforms.fetch_transform(grab.ghost_id).points_local_to_parent(grab.positions)
        real_positions = full_frame.particle_positions[grab.indices]

        target_centroid = target_positions.mean(axis=0)
        real_centroid = real_positions.mean(axis=0)

        self.visuals.update_line(f"{cursor_id}.follow.centroid", positions=[real_centroid, target_centroid], size=0.01, color=[1.0, 0, 0, 1.0])
        self.interactions.update_interaction(f"{cursor_id}.follow.centroid", ParticleInteraction(
            position=target_centroid,
            particles=[int(x) for x in grab.indices],
            type="spring",
            scale=500,
            max_force=100,
        ))

        # rotational following if centroid is close enough
        close = np.linalg.norm(real_centroid - target_centroid, axis=0) < 1

        # find target positions ignoring centroid differences
        rotational_target = target_positions - target_centroid
        rotational_real = real_positions - real_centroid
        rotational = real_positions + (rotational_target - rotational_real)

        for i, index in enumerate(grab.indices):
            if close:
                self.visuals.update_line(f"{cursor_id}.follow.{i}", positions=[rotational[i], real_positions[i]], size=0.01, color=[1.0, 0, 0, 1.0])
                self.interactions.update_interaction(f"{cursor_id}.follow.{i}", ParticleInteraction(
                    position=rotational[i],
                    particles=[int(index)],
                    type="spring",
                    scale=100,
                    max_force=50,
                ))
            else:
                self.visuals.remove_line(f"{cursor_id}.follow.{i}")
                self.interactions.remove_interaction(f"{cursor_id}.follow.{i}")

In [7]:
utilities.show_logging()

Output()